# BirdCLEF 2026 Training v16 — v15 + Pseudo-Labeled Soundscapes
## v15 scored 0.764 (focal recordings only)
## v16 adds soundscape clips whose labels are generated by the v15 ensemble
## Pseudo-label threshold 0.4 | soundscape sample_weight=0.5 in loss
## Config: ResNet18+EfficientNet-B0, 5-fold, fmax=8000, BCE, SpecAugment, 25ep


In [1]:
# === CELL 1: INSTALL & IMPORTS ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, json, ast, copy, random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler  # mixed precision

import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
CFG = dict(
    # Audio
    sr            = 16000,
    seconds       = 5,
    # Mel spectrogram — v7 exact: n_mels=64, n_fft=1024, fmin=60, fmax=sr//2=8000
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,           # v13: v7's default (sr//2), NOT 14000
    # Training
    epochs        = 25,             # v15: +5 epochs for more convergence
    warmup_epochs = 4,              # proportional to 25 epochs
    lr            = 1e-3,           # v7/v12 proven lr for batch=32
    batch_size    = 32,
    patience      = 7,              # v7's patience
    num_workers   = 0,
    seed          = 42,
    # Augmentation
    mixup_alpha   = 0.1,            # v15: lower mixup — less label smoothing noise
    # SpecAugment
    spec_freq_mask = 8,             # v15: mask up to 8 frequency bins
    spec_time_mask = 32,            # v15: mask up to 32 time frames
    # No focal loss — using standard BCE
    # Pseudo-label soundscapes (v16)
    pseudo_label_threshold  = 0.4,  # min ensemble prob to accept a window
    soundscape_sample_weight = 0.5, # downweight pseudo samples in loss
    # Label softening
    secondary_label_weight = 0.3,
    # Architectures — v7's exact pair
    architectures = ["resnet18", "efficientnet_b0"],
    folds         = 5,              # v13: back to v7's 5 folds (10 models total)
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("v16 — v15 + pseudo-labeled soundscapes — imports and config ready")
print(f"   Device       : {CFG['device']}")
print(f"   Folds        : {CFG['folds']}  ({len(CFG['architectures'])*CFG['folds']} total models)")
print(f"   Epochs       : {CFG['epochs']}")
print(f"   Architectures: {CFG['architectures']}")
print(f"   fmax         : {CFG['fmax']} Hz | mixup_alpha={CFG['mixup_alpha']} | specaug=freq{CFG['spec_freq_mask']}/time{CFG['spec_time_mask']}")
print(f"   AMP enabled  : {torch.cuda.is_available()}")


v16 — v15 + pseudo-labeled soundscapes — imports and config ready
   Device       : cuda
   Folds        : 5  (10 total models)
   Epochs       : 25
   Architectures: ['resnet18', 'efficientnet_b0']
   fmax         : 8000 Hz | mixup_alpha=0.1 | specaug=freq8/time32
   AMP enabled  : True


In [2]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TRAIN_META_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/train.csv",
    "/kaggle/input/competitions/birdclef-2026/train.csv",
)
TRAIN_AUDIO_DIR = _first_existing(
    "/kaggle/input/birdclef-2026/train_audio",
    "/kaggle/input/competitions/birdclef-2026/train_audio",
)
TAXONOMY_CSV    = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
SOUNDSCAPE_DIR  = _first_existing(
    "/kaggle/input/birdclef-2026/train_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes",
)
SOUNDSCAPE_ANNO = _first_existing(
    "/kaggle/input/birdclef-2026/train_soundscapes_labels.csv",
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv",
)

# v13: fmax changed to 8000 -> CANNOT reuse v8/v9/v12 mels (different frequency range)
# v15: same mel params as v14 (fmax=8000) — reuse v14 mels if available
# v16: reuse v15/v14 mels (same fmax=8000), pseudo-label mels go into same dir
_v15_mels = "/kaggle/working/mels_v15"
_v14_mels = "/kaggle/working/mels_v14"
if os.path.isdir(_v15_mels) and len(list(Path(_v15_mels).glob("*.npy"))) > 100:
    MEL_OUT_DIR = _v15_mels
    print(f"Reusing v15 mels from {MEL_OUT_DIR}")
elif os.path.isdir(_v14_mels) and len(list(Path(_v14_mels).glob("*.npy"))) > 100:
    MEL_OUT_DIR = _v14_mels
    print(f"Reusing v14 mels from {MEL_OUT_DIR}")
else:
    MEL_OUT_DIR = "/kaggle/working/mels_v16"
    print(f"Computing fresh mels -> {MEL_OUT_DIR}")
    os.makedirs(MEL_OUT_DIR, exist_ok=True)

# v15 checkpoint dir for pseudo-labeling
CKPT_DIR_V15 = _first_existing(
    "/kaggle/input/birdclef-2026-v15-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v15-model",
    "/kaggle/working",
)

print(f"   TRAIN_META_CSV  : {TRAIN_META_CSV}")
print(f"   TAXONOMY_CSV    : {TAXONOMY_CSV}")
print(f"   SOUNDSCAPE_ANNO : {SOUNDSCAPE_ANNO}")

# Load species from taxonomy.csv (authoritative list, 234 species)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
species_set = set(species)
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

df = pd.read_csv(TRAIN_META_CSV)

with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print(f"Loaded {n_classes} species from taxonomy.csv")
print(f"Training samples: {len(df)}, unique species: {df['primary_label'].nunique()}")


Computing fresh mels -> /kaggle/working/mels_v16
   TRAIN_META_CSV  : /kaggle/input/competitions/birdclef-2026/train.csv
   TAXONOMY_CSV    : /kaggle/input/competitions/birdclef-2026/taxonomy.csv
   SOUNDSCAPE_ANNO : /kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv
Loaded 234 species from taxonomy.csv
Training samples: 35549, unique species: 206


In [3]:
# === CELL 3: AUDIO HELPER FUNCTIONS (identical to v8) ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_soft_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype="float32")
    p = str(primary_id)
    if p in sp_idx:
        y[sp_idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[sp_idx[sid]] = CFG["secondary_label_weight"]
    return y

def fixed_length_mono(y: np.ndarray, sr: int, seconds: int = 5) -> np.ndarray:
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")
print(f"   Mel shape: ({CFG['n_mels']}, {int(CFG['sr']*CFG['seconds']/CFG['hop'])+1})")

✅ Helper functions defined
   Mel shape: (64, 251)


In [4]:
# === CELL 4: PRECOMPUTE MELS (skipped if v8 mels already exist) ===
existing = len(list(Path(MEL_OUT_DIR).glob("*.npy")))
if existing > 100 and MEL_OUT_DIR == _v8_mels:
    print(f"♻️  {existing} mels already in {MEL_OUT_DIR} — skipping precompute")
else:
    print(f"Precomputing mels → {MEL_OUT_DIR}")
    failed = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Mel precompute"):
        audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
        mel_name   = row["filename"].replace("/", "_") + ".npy"
        mel_path   = Path(MEL_OUT_DIR) / mel_name
        if mel_path.exists():
            continue
        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
            if sr0 != CFG["sr"]:
                y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
            y   = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(y, CFG["sr"])
            np.save(mel_path, mel)
        except Exception:
            failed += 1
    saved = len(list(Path(MEL_OUT_DIR).glob("*.npy")))
    print(f"✅ Mels ready: {saved} files  ({failed} failed)")

Precomputing mels → /kaggle/working/mels_v16


Mel precompute: 100%|██████████| 35549/35549 [50:52<00:00, 11.65it/s]


✅ Mels ready: 35549 files  (0 failed)


In [5]:
# === CELL 6: LOSS + SPECAUGMENT (v15: BCE replaces focal, SpecAugment added) ===

# v15: standard BCE — focal loss removed
# Focal loss suppresses easy negatives too aggressively for small models on skewed bird data.
criterion = nn.BCEWithLogitsLoss()
print(f"Loss: BCEWithLogitsLoss (no focal)")

# ── SpecAugment ──────────────────────────────────────────────────────────
def spec_augment(mel: torch.Tensor, freq_mask: int, time_mask: int) -> torch.Tensor:
    """Apply frequency and time masking to a (1, n_mels, T) mel tensor."""
    _, F, T = mel.shape
    mel = mel.clone()
    # Frequency masking
    if freq_mask > 0 and F > freq_mask:
        f = random.randint(0, freq_mask)
        f0 = random.randint(0, F - f) if f > 0 else 0
        mel[:, f0:f0 + f, :] = 0.0
    # Time masking
    if time_mask > 0 and T > time_mask:
        t = random.randint(0, time_mask)
        t0 = random.randint(0, T - t) if t > 0 else 0
        mel[:, :, t0:t0 + t] = 0.0
    return mel

print(f"SpecAugment: freq_mask={CFG['spec_freq_mask']} bins, time_mask={CFG['spec_time_mask']} frames")


Loss: BCEWithLogitsLoss (no focal)
SpecAugment: freq_mask=8 bins, time_mask=32 frames


In [6]:
# === CELL 7: MODEL ARCHITECTURES (identical to v8) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = True):
        super().__init__()
        self.arch = arch

        if arch == "resnet18":  # v13: v7's lighter backbone
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch == "resnet50":  # kept for backward compatibility
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

device = torch.device(CFG["device"])
print("✅ BirdCLEFModel defined (timm backbones, pretrained ImageNet)\n")
for arch in CFG["architectures"]:
    m      = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False)
    nparam = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"   {arch:20s}: {nparam:.1f}M parameters")
    del m

✅ BirdCLEFModel defined (timm backbones, pretrained ImageNet)

   resnet18            : 11.6M parameters
   efficientnet_b0     : 4.8M parameters


In [7]:
# === CELL 7: PSEUDO-LABELING SOUNDSCAPES WITH V15 ENSEMBLE ===
# Uses v15 trained models to generate labels for soundscape 5s windows.
# Avoids the broken original approach (CSV had xeno-canto IDs, not species codes).
# Only windows where ensemble probability >= threshold for any species are kept.

_pseudo_done_flag = Path(MEL_OUT_DIR) / '_pseudo_v16_done.txt'

pseudo_df = pd.DataFrame()  # will be concatenated into train_df in Cell 9

if _pseudo_done_flag.exists():
    # Re-run: load previously generated pseudo labels
    _pseudo_csv = Path(MEL_OUT_DIR) / '_pseudo_v16_labels.csv'
    if _pseudo_csv.exists():
        pseudo_df = pd.read_csv(_pseudo_csv)
        print(f'Loaded {len(pseudo_df)} previously pseudo-labeled windows from cache')
    else:
        print('Pseudo-label flag found but CSV missing — will re-generate')
        _pseudo_done_flag.unlink(missing_ok=True)

if not _pseudo_done_flag.exists():
    # Load v15 checkpoints for pseudo-labeling
    _pl_models = []
    _v15_archs = ["resnet18", "efficientnet_b0"]
    _v15_folds = 5
    for _arch in _v15_archs:
        for _fi in range(_v15_folds):
            _ckpt = Path(CKPT_DIR_V15) / f'{_arch}_v15_fold{_fi}.pt'
            if not _ckpt.exists():
                continue
            _m = BirdCLEFModel(_arch, n_classes=n_classes, pretrained=False).to(device)
            _m.load_state_dict(torch.load(str(_ckpt), map_location=device, weights_only=False))
            _m.eval()
            _pl_models.append(_m)
    print(f'Loaded {len(_pl_models)} v15 models for pseudo-labeling')

    _soundscape_dir = _first_existing(
        '/kaggle/input/birdclef-2026/train_soundscapes',
        '/kaggle/input/competitions/birdclef-2026/train_soundscapes',
    )
    _soundscape_files = []
    if os.path.isdir(_soundscape_dir):
        for _ext in ['.ogg', '.wav', '.flac']:
            _soundscape_files.extend(Path(_soundscape_dir).glob(f'*{_ext}'))
    print(f'Found {len(_soundscape_files)} soundscape audio files')

    _threshold  = CFG['pseudo_label_threshold']
    _win_samps  = CFG['sr'] * CFG['seconds']
    _win_frames = int(CFG['seconds'] * CFG['sr'] / CFG['hop']) + 1
    _pseudo_data = []

    def _run_pl_inference(mel_batch_np):
        """Run all v15 models on a batch of mels. Returns mean ensemble probs."""
        if not _pl_models:
            return np.zeros((len(mel_batch_np), n_classes), dtype=np.float32)
        t = torch.from_numpy(mel_batch_np[:, None]).float().to(device)  # (B,1,F,T)
        all_probs = []
        for _m in _pl_models:
            with torch.inference_mode():
                p = torch.sigmoid(_m(t)).float().cpu().numpy()
            all_probs.append(p)
        return np.mean(all_probs, axis=0)  # (B, n_classes)

    if _pl_models and _soundscape_files:
        for _audio_path in tqdm(_soundscape_files, desc='Pseudo-labeling soundscapes'):
            try:
                _wave, _sr0 = sf.read(str(_audio_path), always_2d=False)
            except Exception:
                continue
            if _sr0 != CFG['sr']:
                _wave = librosa.resample(_wave, orig_sr=_sr0, target_sr=CFG['sr'])
            if _wave.ndim == 2:
                _wave = _wave.mean(axis=1)
            _wave = _wave.astype(np.float32)

            # extract all 5s windows from this file
            _n_windows = len(_wave) // _win_samps
            if _n_windows == 0:
                continue

            _mels = []
            for _wi in range(_n_windows):
                _clip = _wave[_wi * _win_samps : (_wi + 1) * _win_samps]
                _mel  = logmel_from_wave(_clip, CFG['sr'])
                T = _mel.shape[1]
                if T < _win_frames:
                    _mel = np.concatenate([_mel, np.zeros((_mel.shape[0], _win_frames - T), dtype=np.float32)], axis=1)
                elif T > _win_frames:
                    _mel = _mel[:, :_win_frames]
                _mels.append(_mel)

            _mels_np = np.stack(_mels)  # (n_windows, n_mels, win_frames)
            _probs   = _run_pl_inference(_mels_np)  # (n_windows, n_classes)

            _base = _audio_path.stem
            for _wi in range(_n_windows):
                _p = _probs[_wi]  # (n_classes,)
                if _p.max() < _threshold:
                    continue  # model not confident about anything — skip
                # top species above threshold
                _top_idx  = np.where(_p >= _threshold)[0]
                _primary  = species[int(_top_idx[np.argmax(_p[_top_idx])])]
                _secondary_list = [species[j] for j in _top_idx if species[j] != _primary]
                _mel_name = f'pseudo_{_base}_w{_wi}'
                np.save(Path(MEL_OUT_DIR) / f'{_mel_name}.npy', _mels_np[_wi])
                _pseudo_data.append({
                    'filename': _mel_name,
                    'primary_label': _primary,
                    'secondary_labels': str(_secondary_list),
                    'sample_weight': CFG['soundscape_sample_weight'],
                })

        if _pseudo_data:
            pseudo_df = pd.DataFrame(_pseudo_data)
            _pseudo_csv = Path(MEL_OUT_DIR) / '_pseudo_v16_labels.csv'
            pseudo_df.to_csv(_pseudo_csv, index=False)
            _pseudo_done_flag.write_text('done')
            print(f'Generated {len(pseudo_df)} pseudo-labeled windows from {len(_soundscape_files)} files')
            print(f'Unique primary species: {pseudo_df["primary_label"].nunique()}')
        else:
            print('No windows passed threshold — training on focal recordings only')
    else:
        if not _pl_models:
            print('WARNING: No v15 checkpoints found. Set CKPT_DIR_V15 correctly.')
        if not _soundscape_files:
            print('WARNING: No soundscape audio files found.')
        print('Proceeding without pseudo-labeled data.')

    # Free pseudo-labeling models from GPU memory
    del _pl_models
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Pseudo-labeled windows ready: {len(pseudo_df)}')


Loaded 10 v15 models for pseudo-labeling
Found 10658 soundscape audio files


Pseudo-labeling soundscapes: 100%|██████████| 10658/10658 [44:43<00:00,  3.97it/s]

Generated 25378 pseudo-labeled windows from 10658 files
Unique primary species: 124
Pseudo-labeled windows ready: 25378


In [8]:
# === CELL 8: DATASET WITH MIXUP (identical to v8) ===
class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, train: bool):
        self.df         = frame.reset_index(drop=True)
        self.mel_root   = Path(mel_root)
        self.train      = train
        self.win_frames = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r        = self.df.iloc[i]
        fname    = str(r["filename"])
        mel_name = (fname if fname.endswith(".npy") else fname.replace("/", "_") + ".npy")
        mel      = np.load(self.mel_root / mel_name).astype("float32")

        T, W = mel.shape[1], self.win_frames
        if T <= W:
            mel = np.concatenate([mel, np.zeros((mel.shape[0], W - T), dtype=np.float32)], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else (T - W) // 2
            mel   = mel[:, start:start + W]

        x = torch.from_numpy(mel[None]).float()
        # v16: SpecAugment applied during training only
        if self.train:
            x = spec_augment(x, CFG["spec_freq_mask"], CFG["spec_time_mask"])
        y = torch.from_numpy(
            row_to_soft_multihot(r["primary_label"], r.get("secondary_labels", "[]"))
        ).float()
        # v16: sample weight (1.0 for focal, 0.5 for pseudo soundscape)
        w = torch.tensor(
            float(r["sample_weight"]) if "sample_weight" in self.df.columns else 1.0
        )
        return x, y, w


def mixup_collate(batch, alpha: float = 0.1, use_mixup: bool = True):
    xs, ys, ws = zip(*batch)
    xs = torch.stack(xs)
    ys = torch.stack(ys)
    ws = torch.stack(ws)
    if not use_mixup or alpha <= 0:
        return xs, ys, ws
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(xs.size(0))
    xs_m = lam * xs + (1 - lam) * xs[idx]
    ys_m = lam * ys + (1 - lam) * ys[idx]
    ws_m = lam * ws + (1 - lam) * ws[idx]  # mix weights proportionally
    return xs_m, ys_m, ws_m


def make_loader(frame, train: bool):
    ds = ClipDataset(frame, MEL_OUT_DIR, train=train)
    return DataLoader(
        ds,
        batch_size=CFG["batch_size"],
        shuffle=train,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: mixup_collate(b, CFG["mixup_alpha"], use_mixup=train),
        drop_last=train,
    )

print("✅ ClipDataset and Mixup collate_fn defined")
print(f"   Mixup alpha = {CFG['mixup_alpha']}  |  Secondary weight = {CFG['secondary_label_weight']}")

✅ ClipDataset and Mixup collate_fn defined
   Mixup alpha = 0.1  |  Secondary weight = 0.3


In [9]:
# === CELL 9: PREPARE TRAINING DATA — focal + pseudo-labeled soundscapes ===
mel_root       = Path(MEL_OUT_DIR)
available_mels = {f.stem for f in mel_root.glob("*.npy")}
print(f"Available mel files: {len(available_mels)}")

train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))
train_df = train_df[train_df["filename"].isin(available_mels)].reset_index(drop=True)
print(f"Training samples from train_audio: {len(train_df)}")

#if len(pseudo_df) > 0:
 #   train_df = pd.concat([train_df, pseudo_df], ignore_index=True)
  #  print(f"+ {len(pseudo_df)} soundscape segments → total: {len(train_df)}")

print(f"\n✅ Training setup:")
print(f"   Total samples : {len(train_df)}")
print(f"   Classes       : {n_classes}")
print(f"   Device        : {device}")

Available mel files: 60927
Training samples from train_audio: 35549

✅ Training setup:
   Total samples : 35549
   Classes       : 234
   Device        : cuda


In [10]:
# === CELL 10: 5-FOLD x 2-ARCH TRAINING (10 total runs) + AMP — v16 ===
n_models = len(CFG["architectures"]) * CFG["folds"]
print("=" * 70)
print(f"TRAINING: {n_models} models  ({CFG['folds']} folds x {len(CFG['architectures'])} architectures)  v16")
print("=" * 70)

_use_amp = (device.type == "cuda")  # v9: AMP enabled on GPU only
print(f"   Mixed precision (AMP) : {'ENABLED' if _use_amp else 'disabled (CPU mode)'}")

_criterion = nn.BCEWithLogitsLoss(reduction='none')  # v16: per-sample weighting
_criterion_mean = nn.BCEWithLogitsLoss()              # for val loss scalar
skf       = StratifiedKFold(n_splits=CFG["folds"], shuffle=True, random_state=CFG["seed"])

oof_preds   = {arch: np.zeros((len(train_df), n_classes), dtype=np.float32)
               for arch in CFG["architectures"]}
arch_scores = {arch: [] for arch in CFG["architectures"]}

for arch in CFG["architectures"]:
    print(f"\n{'='*60}")
    print(f"ARCHITECTURE: {arch}")
    print(f"{'='*60}")

    _label_counts = train_df["primary_label"].value_counts()
    _strat_key    = train_df["primary_label"].apply(
        lambda x: x if _label_counts.get(x, 0) >= CFG["folds"] else "__rare__"
    )
    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(train_df, _strat_key)
    ):
        print(f"\n  Fold {fold_idx + 1}/{CFG['folds']}")

        train_loader = make_loader(train_df.iloc[train_idx], train=True)
        val_loader   = make_loader(train_df.iloc[val_idx],   train=False)

        model     = BirdCLEFModel(arch, n_classes=n_classes, pretrained=True).to(device)
        optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
        scaler    = GradScaler(enabled=_use_amp)  # v9: per-fold AMP scaler

        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                total_iters=CFG["warmup_epochs"])
        cosine_sched = CosineAnnealingLR(optimizer,
                                         T_max=max(1, CFG["epochs"] - CFG["warmup_epochs"]),
                                         eta_min=1e-6)
        scheduler    = SequentialLR(optimizer,
                                    schedulers=[warmup_sched, cosine_sched],
                                    milestones=[CFG["warmup_epochs"]])

        best_val_auc    = -1.0
        patience_count  = 0
        best_state      = None
        best_fold_preds = None

        for epoch in range(CFG["epochs"]):
            # Train
            model.train()
            train_loss = 0.0
            for x, y, w in train_loader:
                x, y, w = x.to(device), y.to(device), w.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    logits_tr = model(x)
                    # weighted BCE: each sample weighted by its sample_weight
                    loss = (_criterion(logits_tr, y).mean(dim=1) * w).mean()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
            train_loss /= max(len(train_loader), 1)
            scheduler.step()

            # Validate
            model.eval()
            val_loss   = 0.0
            fold_preds, fold_targets = [], []
            with torch.no_grad():
                for x, y, _w in val_loader:
                    x, y = x.to(device), y.to(device)
                    with autocast(enabled=_use_amp):
                        logits = model(x)
                    logits_f = logits.float()  # cast fp16->fp32 after autocast
                    val_loss += _criterion_mean(logits_f, y).item()
                    fold_preds.append(torch.sigmoid(logits_f).cpu().numpy())
                    fold_targets.append(y.cpu().numpy())
            val_loss /= max(len(val_loader), 1)

            if not fold_preds:
                # Val loader produced no batches — skip AUC for this epoch
                val_auc = 0.0
            else:
                fp     = np.vstack(fold_preds)
                ft     = np.vstack(fold_targets)
                ft_bin = (ft >= 0.5).astype(np.float32)  # binarise soft targets for AUC
                auc_scores_ep = [
                    roc_auc_score(ft_bin[:, j], fp[:, j])
                    for j in range(n_classes)
                    if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
                ]
                val_auc = np.mean(auc_scores_ep) if auc_scores_ep else 0.0

            cur_lr = scheduler.get_last_lr()[0]

            if val_auc > best_val_auc:
                best_val_auc    = val_auc
                patience_count  = 0
                best_state      = copy.deepcopy(model.state_dict())
                best_fold_preds = fp.copy() if fold_preds else np.zeros((len(val_idx), n_classes), dtype=np.float32)
            else:
                patience_count += 1

            if (epoch + 1) % 5 == 0 or patience_count == 0:
                print(f"    Epoch {epoch+1:3d}: train={train_loss:.4f}  "
                      f"val={val_loss:.4f}  auc={val_auc:.4f}  lr={cur_lr:.2e}")

            if patience_count >= CFG["patience"]:
                print(f"    Early stopping at epoch {epoch+1}")
                break

        # Guard: if best_state was never set (shouldn't happen but be safe)
        if best_state is None:
            print(f"  Warning: no best_state for fold {fold_idx} - saving current weights")
            best_state      = copy.deepcopy(model.state_dict())
            best_fold_preds = np.zeros((len(val_idx), n_classes), dtype=np.float32)

        model.load_state_dict(best_state)
        ckpt_path = f"/kaggle/working/{arch}_v16_fold{fold_idx}.pt"
        torch.save(model.state_dict(), ckpt_path)

        oof_preds[arch][val_idx] = best_fold_preds
        arch_scores[arch].append(best_val_auc)
        print(f"  Fold {fold_idx+1} best AUC: {best_val_auc:.4f}  saved {ckpt_path}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_auc = np.mean(arch_scores[arch])
    print(f"\n  {arch}: mean OOF AUC = {mean_auc:.4f}")

print(f"\nAll {n_models} models trained!")

TRAINING: 10 models  (5 folds x 2 architectures)  v16
   Mixed precision (AMP) : ENABLED

ARCHITECTURE: resnet18

  Fold 1/5


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/tmp/ipykernel_23/1849750891.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=_use_amp)  # v9: per-fold AMP scaler
/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):


    Epoch   1: train=0.0575  val=0.0269  auc=0.5300  lr=3.25e-04


/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):


...

    Epoch  15: train=0.0082  val=0.0097  auc=0.9734  lr=4.63e-04


/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.p

    Epoch  20: train=0.0060  val=0.0096  auc=0.9721  lr=1.34e-04


/tmp/ipykernel_23/1849750891.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):
/tmp/ipykernel_23/1849750891.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=_use_amp):


    Early stopping at epoch 21
  Fold 5 best AUC: 0.9770  saved /kaggle/working/efficientnet_b0_v16_fold4.pt

  efficientnet_b0: mean OOF AUC = 0.9755

All 10 models trained!


In [11]:
# === CELL 11: OOF ENSEMBLE AUC & SUMMARY ===
print("=" * 70)
print("TRAINING SUMMARY v16")
print("=" * 70)

for arch in CFG["architectures"]:
    fold_aucs = arch_scores[arch]
    print(f"\n📊 {arch}:")
    print(f"   Fold AUCs : {[f'{a:.4f}' for a in fold_aucs]}")
    print(f"   Mean AUC  : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

ensemble_oof = np.mean([oof_preds[a] for a in CFG["architectures"]], axis=0)

oof_targets = np.zeros((len(train_df), n_classes), dtype=np.float32)
for i, row in train_df.iterrows():
    oof_targets[i] = row_to_soft_multihot(row["primary_label"], row.get("secondary_labels", "[]"))
oof_targets_bin = (oof_targets >= 0.5).astype(np.float32)

ensemble_auc_scores = [
    roc_auc_score(oof_targets_bin[:, j], ensemble_oof[:, j])
    for j in range(n_classes)
    if oof_targets_bin[:, j].sum() > 0 and (1 - oof_targets_bin[:, j]).sum() > 0
]
print(f"\n🏆 {len(CFG['architectures'])}-Architecture OOF Macro AUC: {np.mean(ensemble_auc_scores):.4f}")

print(f"\n✅ Saved checkpoints:")
for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        print(f"   /kaggle/working/{arch}_v12_fold{fold_idx}.pt")

TRAINING SUMMARY v16

📊 resnet18:
   Fold AUCs : ['0.9695', '0.9729', '0.9696', '0.9718', '0.9721']
   Mean AUC  : 0.9712 ± 0.0014

📊 efficientnet_b0:
   Fold AUCs : ['0.9745', '0.9745', '0.9767', '0.9747', '0.9770']
   Mean AUC  : 0.9755 ± 0.0011

🏆 2-Architecture OOF Macro AUC: 0.9731

✅ Saved checkpoints:
   /kaggle/working/resnet18_v12_fold0.pt
   /kaggle/working/resnet18_v12_fold1.pt
   /kaggle/working/resnet18_v12_fold2.pt
   /kaggle/working/resnet18_v12_fold3.pt
   /kaggle/working/resnet18_v12_fold4.pt
   /kaggle/working/efficientnet_b0_v12_fold0.pt
   /kaggle/working/efficientnet_b0_v12_fold1.pt
   /kaggle/working/efficientnet_b0_v12_fold2.pt
   /kaggle/working/efficientnet_b0_v12_fold3.pt
   /kaggle/working/efficientnet_b0_v12_fold4.pt
